# Distributing Gradient Descent
Author: Tomasz Kanas

In this class we will learn several methods to distribute Gradient Descent.

We implement the algorithms in PyTorch and run them on the cloud setup we have created last week, using MPI runner.

## Setup cloud environment

You will need the cloud environment we have created last week to complete this class. If you haven't done it yet, firstly finish [this](https://drive.google.com/file/d/1XKDSevaX6SnxbtxlQI5ZjP3dcMdy8HeH/view?usp=sharing) scenario.

To recover the environment you just have to run the following commands in the directory you placed the `simple_deployment.tf` file. Note that the execution may take several minutes.
```sh
. ./setup.sh
terraform apply
python parse-tf-state.py
ansible-playbook -i hosts install_packages.yml
ansible-playbook -i hosts config_ssh.yml
ansible-playbook -i hosts nfs.yml
```

To test the setup copy the master VM IP (the one printed by terraform) to the commands below and run them:

```
export GCP_IP=VM_ID
scp -i .ssh/id_gcp reduce.py $GCP_userID@$GCP_IP:~
ssh -i .ssh/id_gcp $GCP_userID@$GCP_IP "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 reduce.py"
```

## Running MPI tasks from Colab
This step is optional. For your convenience we can setup a way to run code from colab on your cloud VMs.

Firstly, we need to create new ssh key-pair on the colab. Set the value of `GCP_userID` to the username you are using to connect to your VMs, and run the cells below.

In [1]:
%env GCP_userID=pilkiewiczkamil

env: GCP_userID=pilkiewiczkamil


In [ ]:
!mkdir .ssh
!ssh-keygen -t ed25519 -f .ssh/id_gcp -N '' -C $GCP_userID
!cat .ssh/id_gcp.pub

Generating public/private ed25519 key pair.
Your identification has been saved in .ssh/id_gcp
Your public key has been saved in .ssh/id_gcp.pub
The key fingerprint is:
SHA256:BN9hOOW5w0mlCY3ge80wfXEdKMSfhwUtfywbsNA50/U pilkiewiczkamil
The key's randomart image is:
+--[ED25519 256]--+
|      o..=*oo+=+o|
|     . o+*oO*=.oo|
|      . =.O.+=*.E|
|       o B +.+ooo|
|      . S B   .+.|
|       .   .  .  |
|                 |
|                 |
|                 |
+----[SHA256]-----+
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIDg7JIyl1J3yy+ov1pYN5lrfHX7NQywMSKZ0nX2Ufrvs pilkiewiczkamil


The last line of the output is the public key, add it to your cloud metadata:
```
Hamburger (☰) -> Compute Engine -> Metadata
Navigate to 'SSH KEYS' -> 'Edit' -> 'Add Item'
```

NOTE: The files on colab are not persistent, so you will have to repeat those steps every time the colab environment disconnects. To check if you still have the keys, you can use the colab file explorer (folder icon on the left side of the screen). You should see the `.ssh` folder with two key files: `id_gcp` and `id_gcp.pub`.

Now we define the function that will run the `scp` and `ssh` commands from colab. Paste the `key_node` IP from hour `hosts` file to the `MASTER_IP` variable and run the cell.

In [ ]:
import os
import sys
import argparse
import subprocess

# TODO: paste the key_node IP from your hosts file here
MASTER_IP = "34.68.45.15"

def mpi_run_python(script, mpi_flags):
    guser = os.environ['GCP_userID']

    # copy source file to one of the nodes
    print(f"\n ### copy source code to node {MASTER_IP} ###")
    command = f"scp -i .ssh/id_gcp {script} {guser}@{MASTER_IP}:./"
    print(command)
    p = subprocess.run(command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    p.check_returncode()

    # Exec the mpi program
    print("\n ### Running MPI ###")
    command = f"ssh -i .ssh/id_gcp {guser}@{MASTER_IP} \"mpiexec {mpi_flags} python3 {script}\""
    print(command)
    p = subprocess.run(command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)
    p.check_returncode()
    # Useful command for debugging
    # os.system("ssh -i {0} {1}@{2} \"{3} --hostfile hostfile_mpi --mca btl_base_warn_component_unused 0 ./{4}\"".format(".ssh/id_gcp", guser, MASTER_IP, nb_ps, execfile))


IMPORTANT: The IPs will be different every time you recreate the VMs, so you need to change the `MASTER_IP` in the script above everytime you recreate the VMs.

Although, sometimes when you recreate the VMs they will accidentaly get the same IPs what causes problems with SSH security mechanisms (as the public keys of those VMs will change). To be able to troubleshoot such scenario, run the following cell:

In [ ]:
!mkdir /root/.ssh/
!touch /root/.ssh/known_hosts

To run the script above we need to create a file with the script we want to run. It can be done on Colab using the `%%file filename` magic string at the beggining of the cell - then execution will save the cell contents to the file instead of running it. To test this, let's use the example from the previous class:

In [ ]:
%%file reduce.py
import os
import time
import torch
import torch.distributed as dist

WORLD_SIZE = int(os.environ['OMPI_COMM_WORLD_SIZE'])
RANK = int(os.environ['OMPI_COMM_WORLD_RANK'])
dist.init_process_group("gloo", rank=RANK, world_size=WORLD_SIZE)

message = torch.tensor(RANK)
print(f"Rank {RANK}: Sending {message.item()}")
dist.reduce(message, 0, dist.ReduceOp.SUM)

if RANK == 0:
  print(f"Rank 0: Sum of all messages is {message.item()}")

Writing reduce.py


In [ ]:
# Students cell:
!pwd
!ls -a
!ls .ssh
!cat .ssh/id_gcp.pub
# !cat .ssh/id_gcp

/content
.  ..  .config	reduce.py  sample_data	.ssh
id_gcp	id_gcp.pub
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIDg7JIyl1J3yy+ov1pYN5lrfHX7NQywMSKZ0nX2Ufrvs pilkiewiczkamil


Now we can run it by executing:

In [ ]:
mpi_run_python("reduce.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp reduce.py pilkiewiczkamil@34.68.45.15:./
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 reduce.py"
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
Rank 0: Sending 0
Rank 1: Sending 1
Rank 2: Sending 2
Rank 0: Sum of all messages is 3



If you receive an error message: "Host key authentication failed", try those commands (with the IP of the machine the script is sending the code to)

In [ ]:
!ssh-keygen -f "/root/.ssh/known_hosts" -R "34.68.45.15"
!ssh-keyscan 34.68.45.15 | tee -a ~/.ssh/known_hosts

Host 34.68.45.15 not found in /root/.ssh/known_hosts
# 34.68.45.15:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u5
# 34.68.45.15:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u5
# 34.68.45.15:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u5
# 34.68.45.15:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u5
# 34.68.45.15:22 SSH-2.0-OpenSSH_8.4p1 Debian-5+deb11u5
34.68.45.15 ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIIhVS7KA9xIHffFo0D1fNZS5SKnzCSCshPsyiv2zD1r/
34.68.45.15 ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAABgQDC3TEKqcCajIoxadYWvPhPrPx1BhId6SsPG2XU/Fj4i2G/DVuNhMU92J+G+FB5FQs/M+qhVJbgjIk0n48vwEbXH1ZYW5f+DTVAMRYrJqds3b1sqy6vJFn0EXSZjIkMeBdrRx8bVgDNZC1MG7Xl5MGN2AVDIm7Wkb/bHxIJfJs5/Tnj2KTf8GY7nEeyjg/lff3gGMFvobGzU7FuI3nioyQqRT2jrjVRMHupfVacWyAUJPy21SxGsRe78lkwCrLIcn03hhhJv04dxNURfqQQisERz0xO1Xd83Kof4jKQcxEY8hAyoxF8b1++AhqDOsONmWFPE6TSRn6cuFCu6gv1bga7EEWPyqTBZek94aSKblpQ4Et+CFMfwNsgLB3IaSftlG9HDJ6iRsVBLHmgntza6VxSIxueHRfWUylMMQs1rC2wlqoZDJYroExX627NZp03avyojdOL8tH42wMy23fBGhspSWb0lOB9GEJrOuFVKHT8dpseQyW2ocQQ95908bmhYdE=
34.68.45.

If you see
```
bash: warning: setlocale: LC_ALL: cannot change locale en_US.UTF-8)
```
in the output, and it irritates you, run the cell below.


In [ ]:
%env LC_ALL=C

env: LC_ALL=C


## Distributing the Gradient Descent

### Model: multivariate linear regression

For simplicity we will use a multivariate linear regression model, and for data, we will train it on a random data sampled from Normal distribution. Below you have a simple sequential implementation of gradient descent for this model, that we will use as a baseline.

In [ ]:
import numpy as np
import torch

np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)

SEED = 42
np.random.seed(SEED)

TRAINING_DATASET_SIZE = 3000
FEATURES = 6

X_numpy = np.random.multivariate_normal(np.arange(FEATURES), np.eye(FEATURES), TRAINING_DATASET_SIZE)
X_train = torch.tensor(X_numpy).reshape(TRAINING_DATASET_SIZE, FEATURES)
# The hidden coefficients, that model is supposed to learn
COEFFICIENTS = np.array([6, 5, 4, 3, 2, 1], dtype=np.float64)
# Matrix multiplication
y_numpy = X_numpy @ COEFFICIENTS
y_train = torch.tensor(y_numpy)

#Initial solution
w_numpy = np.random.randn(FEATURES)

def forward(w, X):
  return (X @ w).reshape(X.shape[0])

def mse(y, y_est):
  err = y - y_est
  return (err ** 2).mean()

Now we can run simple training loop

In [ ]:
EPOCHS = 200
LEARNING_RATE = 0.01

w = torch.tensor(w_numpy, requires_grad = True)
loss = torch.tensor([0.0], requires_grad = True)

for epoch in range(EPOCHS):
  y_est = forward(w, X_train)
  loss = mse(y_train, y_est)
  loss.backward()

  with torch.no_grad():
    w -= LEARNING_RATE * w.grad
  w.grad.detach_()
  w.grad.zero_()

w.detach().numpy()

array([5.909, 4.915, 3.942, 2.979, 2.03 , 1.029])

Note, that with the provided parameters, the iteration has *almost* converged - those numbers we can use later to verify, that our distributed algorithms produce exactly the same results.

The remainder of this lab aims at exploring different methods of distributing gradient descent. Actually, as you will quickly realize, the only difficult step is the implementation of the `all_reduce` method, and all the metods we will see, will be in fact different algorithms for `all_reduce`.

### Gradient Accumulation

We will start with very simple concept:
1. Distribute data equally among the nodes.
2. Perform one batch (or epoch in our case, as we have 1 batch per epoch) on each machine.
3. Sum the gradients: use the `reduce` method to compute the sum of gradients in the 0-th node, and then `broadcast`, to broadcast the accumulated gradients to all of the nodes.
4. Repeat this for every batch.

The PyTorch Distributed [documentation](https://pytorch.org/docs/stable/distributed.html) may be handy for those tasks. You can also take a look on the first lab scenario.

Implement this algorithm by extending the code cell below.

NOTE: Pytorch is very sensitive with dimensions and types, so make sure that the types and dimensions of the tensors you send and those you receive match.


In [ ]:
%%file sgd_accumulate.py
import os
import numpy as np
import torch
import torch.distributed as dist

np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)

def forward(w, X):
  return (X @ w).reshape(X.shape[0])

def mse(y, y_est):
  err = y - y_est
  return (err ** 2).mean()

WORLD_SIZE = int(os.environ['OMPI_COMM_WORLD_SIZE'])
RANK = int(os.environ['OMPI_COMM_WORLD_RANK'])
dist.init_process_group("gloo", rank=RANK, world_size=WORLD_SIZE)

SEED = 42
np.random.seed(SEED)

TRAINING_DATASET_SIZE = 3000
FEATURES = 6

LOCAL_DATASET_SIZE = int(TRAINING_DATASET_SIZE / WORLD_SIZE)

scatter_list = None
w = torch.empty(FEATURES, dtype=torch.float64)
if RANK == 0:
  #Let's suppose that we start with all the data on one machine
  X_numpy = np.random.multivariate_normal(np.arange(FEATURES), np.eye(FEATURES), TRAINING_DATASET_SIZE)
  X_train = torch.tensor(X_numpy).reshape(TRAINING_DATASET_SIZE, FEATURES)
  # The hidden coefficients, that model is supposed to learn
  COEFFICIENTS = np.array([6, 5, 4, 3, 2, 1], dtype=np.float64)
  # Matrix multiplication
  y_numpy = X_numpy @ COEFFICIENTS
  y_train = torch.tensor(y_numpy)

  w_numpy = np.random.randn(FEATURES)
  w = torch.tensor(w_numpy, dtype=torch.float64)
  # Append y to X to send everything in one batch
  data = torch.cat((X_train, y_train[:,None]), dim=1)
  # Divide the data into equal parts that will be sent
  scatter_list = [data[i * LOCAL_DATASET_SIZE : (i + 1) * LOCAL_DATASET_SIZE, :] for i in range(WORLD_SIZE)]

data = torch.empty((LOCAL_DATASET_SIZE, (FEATURES + 1)), dtype=torch.float64) # allocate space for received data - need FEATURES columns for features, and one for y
# Distribute data to all workers
dist.scatter(data, scatter_list, src=0)
# Broadcast the initial state
dist.broadcast(w, src=0)
data = data.reshape(LOCAL_DATASET_SIZE, FEATURES + 1)
X_train = data[:,:-1]
y_train = data[:,-1]

EPOCHS = 200
LEARNING_RATE = 0.01

w.requires_grad = True
loss = torch.tensor([0.0], requires_grad = True)

for epoch in range(EPOCHS):
  #TODO: implement the algorithm
  loss = mse(y_train, forward(w, X_train))
  loss.backward()
  dist.reduce(w.grad, 0, dist.ReduceOp.SUM)
  if RANK == 0:
    w.grad /= WORLD_SIZE
  dist.broadcast(w.grad, 0)
  with torch.no_grad():
    w -= LEARNING_RATE * w.grad
  w.grad.detach_()
  w.grad.zero_()

  # TODO END

#At the end every node should have the same coefficients
print(f"rank = {RANK}, data = {np.array2string(w.detach().numpy())}")

Writing sgd_accumulate.py


In [ ]:
mpi_run_python("sgd_accumulate.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp sgd_accumulate.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 sgd_accumulate.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 2, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 1, data = [5.909 4.915 3.942 2.979 2.03  1.029]



You should see the same state in every node at the end, and the state should match exactly the one in sequential version.

### Parallelizing the communication of the gradients
You might have noticed, that the previous algorithm has problem with scalability: every machine sends its gradient to the machine with rank 0, and then this machine sends accumulated gradient to all machines.

Try to figure it out how we can do this more efficiently before reading the next cell.

#### Algorithm

We will place the machines on the cycle and each machine will send its gradient to the next machine on the cycle (and receive from the previous one).

Then each machine will add its gradient to the received gradient, and in the next iteration will send the sum of the gradients.

We need to perform the same number of iterations as we have machines minus one.

For visualization, see the image below.

<img src="https://i.imgur.com/vHfxzQt.png"/>

Implement this algorithm below. Note that the [`send`](https://mpi4py.readthedocs.io/en/stable/reference/mpi4py.MPI.Comm.html?highlight=Send#mpi4py.MPI.Comm.Send) and [`recv`](https://mpi4py.readthedocs.io/en/stable/reference/mpi4py.MPI.Message.html?highlight=Recv#mpi4py.MPI.Message.Recv) methods are blocking, so to avoid deadlock you need either:
- On even nodes first `send` then `recv` and on the odd nodes first `recv` then `send`.
- Use the asynchronous operations `isend` and `irecv`.
- Use the [`batch_isend_irecv`](https://pytorch.org/docs/stable/distributed.html#torch.distributed.batch_isend_irecv) method.

In [ ]:
%%file sgd_cycle.py
import os
import numpy as np
import torch
import torch.distributed as dist

np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)

def forward(w, X):
  return (X @ w).reshape(X.shape[0])

def mse(y, y_est):
  err = y - y_est
  return (err ** 2).mean()

WORLD_SIZE = int(os.environ['OMPI_COMM_WORLD_SIZE'])
RANK = int(os.environ['OMPI_COMM_WORLD_RANK'])
dist.init_process_group("gloo", rank=RANK, world_size=WORLD_SIZE)

SEED = 42
np.random.seed(SEED)

TRAINING_DATASET_SIZE = 3000
FEATURES = 6

LOCAL_DATASET_SIZE = int(TRAINING_DATASET_SIZE / WORLD_SIZE)

scatter_list = None
w = torch.empty(FEATURES, dtype=torch.float64)
if RANK == 0:
  #Let's suppose that we start with all the data on one machine
  X_numpy = np.random.multivariate_normal(np.arange(FEATURES), np.eye(FEATURES), TRAINING_DATASET_SIZE)
  X_train = torch.tensor(X_numpy).reshape(TRAINING_DATASET_SIZE, FEATURES)
  # The hidden coefficients, that model is supposed to learn
  COEFFICIENTS = np.array([6, 5, 4, 3, 2, 1], dtype=np.float64)
  # Matrix multiplication
  y_numpy = X_numpy @ COEFFICIENTS
  y_train = torch.tensor(y_numpy)

  w_numpy = np.random.randn(FEATURES)
  w = torch.tensor(w_numpy, dtype=torch.float64)
  # Append y to X to send everything in one batch
  data = torch.cat((X_train, y_train[:,None]), dim=1)
  # Divide the data into equal parts that will be sent
  scatter_list = [data[i * LOCAL_DATASET_SIZE : (i + 1) * LOCAL_DATASET_SIZE, :] for i in range(WORLD_SIZE)]

data = torch.empty((LOCAL_DATASET_SIZE, (FEATURES + 1)), dtype=torch.float64) # allocate space for received data - need FEATURES columns for features, and one for y
# Distribute data to all workers
dist.scatter(data, scatter_list, src=0)
# Broadcast the initial state
dist.broadcast(w, src=0)
data = data.reshape(LOCAL_DATASET_SIZE, FEATURES + 1)
X_train = data[:,:-1]
y_train = data[:,-1]

EPOCHS = 200
LEARNING_RATE = 0.01

w.requires_grad = True
loss = torch.tensor([0.0], requires_grad = True)

for epoch in range(EPOCHS):
  #TODO: implement the algorithm
  loss = mse(y_train, forward(w, X_train))
  loss.backward()

  w_grad_acc = torch.zeros(FEATURES, dtype=torch.float64)
  temp = torch.zeros(FEATURES, dtype=torch.float64)
  prev = (RANK - 1 + WORLD_SIZE) % WORLD_SIZE
  next = (RANK + 1) % WORLD_SIZE
  wgrad = w.grad.detach()

  for i in range(WORLD_SIZE-1):
    p = w_grad_acc + wgrad # WHY THIS IS CRUCIAL????
    if RANK % 2 == 0:
      dist.send(p, next)
      dist.recv(w_grad_acc, prev) # You get info only from one source.
    else:
      dist.recv(temp, prev)
      dist.send(p, next)
      w_grad_acc = temp
  wgrad += w_grad_acc
  wgrad /= WORLD_SIZE

  with torch.no_grad():
    w -= LEARNING_RATE * wgrad
  w.grad.detach_()
  w.grad.zero_()
  # TODO END

#At the end every node should have the same coefficients
print(f"rank = {RANK}, data = {np.array2string(w.detach().numpy())}")

Overwriting sgd_cycle.py


In [ ]:
mpi_run_python("sgd_cycle.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp sgd_cycle.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 sgd_cycle.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 1, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 2, data = [5.909 4.915 3.942 2.979 2.03  1.029]



Again, verify that all nodes have the same coefficients at the end, and those coefficients are equal to the ones produced by the sequential solution.

### Ring AllReduce algorithm

Ring AllReduce algorithm goes one step further than the previous algorithm. In large models, sending whole gradient in one round may be too much for you bandwidth capacity. In the Ring AllReduce algorithm, in one round we will send only one chunk of the gradient. Because of that, we will actually need two full cycles of communication to distribute all the gradients among all the nodes.

The algorithm looks like this:
1. Compute forward and backward pass on the local part of the data.
2. Split the gradient into the same number of chunks, you have workers in the network.
3. In `i`-th iteration the worker with rank `RANK` sends the accumulated chunk number `RANK - i - 1` (the one it received in the previous iteration) to the next node. We do `WORLD_SIZE - 1` of such iterations.
4. After `WORLD_SIZE - 1` such iterations, each node has one fully aggregated data chunk.
5. Now we do another round on the cycle, to distribute the fully aggregated data chunks among the nodes.

See the animation below for visualization.

In [ ]:
%%html
<video  width="960" height="720" autoplay loop muted playsinline controls>
    <source src="https://i.imgur.com/IV6jBwL.mp4" type="video/mp4">
</video>

Implement this algorihtm below.

In [ ]:
%%file horovod.py
import os
import numpy as np
import torch
import torch.distributed as dist

np.set_printoptions(precision=3)
np.set_printoptions(suppress=True)

def forward(w, X):
  return (X @ w).reshape(X.shape[0])

def mse(y, y_est):
  err = y - y_est
  return (err ** 2).mean()

WORLD_SIZE = int(os.environ['OMPI_COMM_WORLD_SIZE'])
RANK = int(os.environ['OMPI_COMM_WORLD_RANK'])
dist.init_process_group("gloo", rank=RANK, world_size=WORLD_SIZE)

SEED = 42
np.random.seed(SEED)

TRAINING_DATASET_SIZE = 3000
FEATURES = 6

LOCAL_DATASET_SIZE = int(TRAINING_DATASET_SIZE / WORLD_SIZE)

scatter_list = None
w = torch.empty(FEATURES, dtype=torch.float64)
if RANK == 0:
  #Let's suppose that we start with all the data on one machine
  X_numpy = np.random.multivariate_normal(np.arange(FEATURES), np.eye(FEATURES), TRAINING_DATASET_SIZE)
  X_train = torch.tensor(X_numpy).reshape(TRAINING_DATASET_SIZE, FEATURES)
  # The hidden coefficients, that model is supposed to learn
  COEFFICIENTS = np.array([6, 5, 4, 3, 2, 1], dtype=np.float64)
  # Matrix multiplication
  y_numpy = X_numpy @ COEFFICIENTS
  y_train = torch.tensor(y_numpy)

  w_numpy = np.random.randn(FEATURES)
  w = torch.tensor(w_numpy, dtype=torch.float64)
  # Append y to X to send everything in one batch
  data = torch.cat((X_train, y_train[:,None]), dim=1)
  # Divide the data into equal parts that will be sent
  scatter_list = [data[i * LOCAL_DATASET_SIZE : (i + 1) * LOCAL_DATASET_SIZE, :] for i in range(WORLD_SIZE)]

data = torch.empty((LOCAL_DATASET_SIZE, (FEATURES + 1)), dtype=torch.float64) # allocate space for received data - need FEATURES columns for features, and one for y
# Distribute data to all workers
dist.scatter(data, scatter_list, src=0)
# Broadcast the initial state
dist.broadcast(w, src=0)
data = data.reshape(LOCAL_DATASET_SIZE, FEATURES + 1)
X_train = data[:,:-1]
y_train = data[:,-1]

EPOCHS = 200
LEARNING_RATE = 0.01

w.requires_grad = True
loss = torch.tensor([0.0], requires_grad = True)

for epoch in range(EPOCHS):
  #TODO: implement the algorithm
  loss = mse(y_train, forward(w, X_train))
  loss.backward()
  wgrad = w.grad.clone().detach()
  w_grad_acc = torch.zeros(FEATURES, dtype=torch.float64)
  package_size = FEATURES // WORLD_SIZE # We abuse the fact, that features%world_size=0.
  temp = torch.zeros(package_size, dtype=torch.float64)
  prev = (RANK - 1 + WORLD_SIZE) % WORLD_SIZE
  next = (RANK + 1) % WORLD_SIZE
  # Communication start:
  # Reduce faze:
  for i in range(WORLD_SIZE-1):
    p_id = (RANK-i+WORLD_SIZE)%WORLD_SIZE
    from_ = p_id*package_size
    to_ = from_ + package_size
    p = w_grad_acc[from_:to_] + wgrad[from_:to_]
    if RANK % 2 == 0:
      dist.send(p, next)
      dist.recv(temp, prev)
    else:
      dist.recv(temp, prev)
      dist.send(p, next)
    # Remember: p is in different place than temp in w_grad_acc, temp is earlier.
    temp_id = (RANK-i-1+WORLD_SIZE)%WORLD_SIZE
    from_ = temp_id*package_size
    to_ = from_ + package_size
    w_grad_acc[from_:to_] = temp
  from_ = RANK*package_size
  to_ = from_ + package_size
  wgrad[from_:to_] += w_grad_acc[from_:to_]
  wgrad[from_:to_] /= WORLD_SIZE
  # Broadcast faze:
  for i in range(WORLD_SIZE-1):
    p_id = (RANK-i+WORLD_SIZE)%WORLD_SIZE
    from_ = p_id*package_size
    to_ = from_ + package_size
    p = wgrad[from_:to_]
    if RANK % 2 == 0:
      dist.send(p, next)
      dist.recv(temp, prev)
    else:
      dist.recv(temp, prev)
      dist.send(p, next)
    temp_id = (RANK-i-1+WORLD_SIZE)%WORLD_SIZE
    from_ = temp_id*package_size
    to_ = from_ + package_size
    wgrad[from_:to_] = temp
  # End of communication
  with torch.no_grad():
    w -= LEARNING_RATE * wgrad
  w.grad.detach_()
  w.grad.zero_()
  # TODO END

#At the end every node should have the same coefficients
print(f"rank = {RANK}, data = {np.array2string(w.detach().numpy())}")

Overwriting horovod.py


In [ ]:
mpi_run_python("horovod.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp horovod.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [4.323 3.525 2.943 2.117 2.576 1.808]
rank = 1, data = [4.323 3.525 2.943 2.117 2.576 1.808]
rank = 2, data = [4.323 3.525 2.943 2.117 2.576 1.808]



## Excercise

Measure the training time for several (at least 4) different numbers of features for all 3 presented algorithms. What can you deduce from the results?

In [ ]:
def change_features(fid, filename):
  with open(f"{filename}.py", "r", encoding="utf-8") as in_:
    with open(f"{filename}{fid}.py", "w", encoding="utf-8") as out_:
        for line in in_:
            if "FEATURES =" in line:
                nl = f"FEATURES = {6*fid}"
            elif "COEFFICIENTS =" in line:
               list_ = [fid*6 - i for i in range(fid*6)]
               nl = f"  COEFFICIENTS = np.array({list_}, dtype=np.float64)"
            else:
                nl = line
            # Zapisujemy do nowego pliku
            out_.write(nl)
for fid in range(1,5):
  change_features(fid, "horovod")
  change_features(fid, "sgd_accumulate")
  change_features(fid, "sgd_cycle")

In [ ]:
%%time
mpi_run_python("horovod1.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp horovod1.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod1.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [4.323 3.525 2.943 2.117 2.576 1.808]
rank = 1, data = [4.323 3.525 2.943 2.117 2.576 1.808]
rank = 2, data = [4.323 3.525 2.943 2.117 2.576 1.808]

CPU times: user 2.71 ms, sys: 1.99 ms, total: 4.7 ms
Wall time: 7.05 s


In [ ]:
%%time
mpi_run_python("horovod2.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp horovod2.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod2.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [-2.762e+73 -6.119e+74 -1.263e+75 -1.892e+75 -2.553e+75 -3.177e+75
 -3.768e+75 -4.431e+75 -5.056e+75 -5.665e+75 -6.309e+75 -6.954e+75]
rank = 1, data = [-2.762e+73 -6.119e+74 -1.263e+75 -1.892e+75 -2.553e+75 -3.177e+75
 -3.768e+75 -4.431e+75 -5.056e+75 -5.665e+75 -6.309e+75 -6.954e+75]
rank = 2, data = [-2.762e+73 -6.119e+74 -1.263e+75 -1.892e+75 -2.553e+75 -3.177e+75
 -3.768e+75 -4.431e+75 -5.056e+75 -5.665e+75 -6.309e+75 -6.

In [ ]:
%%time
mpi_run_python("horovod3.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp horovod3.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod3.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [-1.426e+205 -2.295e+207 -4.484e+207 -6.765e+207 -9.008e+207 -1.133e+208
 -1.343e+208 -1.582e+208 -1.812e+208 -2.032e+208 -2.262e+208 -2.486e+208
 -2.735e+208 -2.951e+208 -3.178e+208 -3.394e+208 -3.641e+208 -3.874e+208]
rank = 2, data = [-1.426e+205 -2.295e+207 -4.484e+207 -6.765e+207 -9.008e+207 -1.133e+208
 -1.343e+208 -1.582e+208 -1.812e+208 -2.032e+208 -2.262e+208 -2.486e+208
 -2.735e+208 -2.951e+208 -3.178e+208 -3.394e+20

In [ ]:
%%time
mpi_run_python("horovod4.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp horovod4.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod4.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [-1.840e+285 -3.858e+288 -7.809e+288 -1.173e+289 -1.559e+289 -1.963e+289
 -2.312e+289 -2.717e+289 -3.103e+289 -3.479e+289 -3.876e+289 -4.263e+289
 -4.640e+289 -5.043e+289 -5.455e+289 -5.818e+289 -6.220e+289 -6.610e+289
 -6.995e+289 -7.369e+289 -7.754e+289 -8.155e+289 -8.541e+289 -8.935e+289]
rank = 1, data = [-1.840e+285 -3.858e+288 -7.809e+288 -1.173e+289 -1.559e+289 -1.963e+289
 -2.312e+289 -2.717e+289 -3.103e+289 -3.479e+28

In [ ]:
%%time
mpi_run_python("sgd_accumulate1.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp sgd_accumulate1.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 sgd_accumulate1.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 2, data = [5.909 4.915 3.942 2.979 2.03  1.029]
rank = 1, data = [5.909 4.915 3.942 2.979 2.03  1.029]

CPU times: user 3.85 ms, sys: 170 µs, total: 4.02 ms
Wall time: 7.52 s


In [ ]:
%%time
mpi_run_python("sgd_accumulate2.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp sgd_accumulate2.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 sgd_accumulate2.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [-4.000e+189 -6.696e+191 -1.360e+192 -2.042e+192 -2.727e+192 -3.405e+192
 -4.066e+192 -4.778e+192 -5.446e+192 -6.106e+192 -6.804e+192 -7.479e+192]
rank = 2, data = [-4.000e+189 -6.696e+191 -1.360e+192 -2.042e+192 -2.727e+192 -3.405e+192
 -4.066e+192 -4.778e+192 -5.446e+192 -6.106e+192 -6.804e+192 -7.479e+192]
rank = 1, data = [-4.000e+189 -6.696e+191 -1.360e+192 -2.042e+192 -2.727e+192 -3.405e+192
 -4.066e+192 -4

In [ ]:
%%time
mpi_run_python("sgd_accumulate3.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp sgd_accumulate3.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 sgd_accumulate3.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
rank = 2, data = [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]
rank = 1, data = [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan]

CPU times: user 2.89 ms, sys: 1.7 ms, total: 4.59 ms
Wall time: 6.87 s


In [ ]:
%%time
mpi_run_python("sgd_accumulate4.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp sgd_accumulate4.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 sgd_accumulate4.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan]
rank = 2, data = [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan]
rank = 1, data = [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan]

CPU times: user 4.38 ms, sys: 0 ns, total: 4.38 ms
Wall time: 5.35 s


In [ ]:
%%time
mpi_run_python("sgd_accumulate1.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")

In [ ]:
change_features(100, "horovod")
change_features(200, "horovod")
change_features(400, "horovod")
change_features(800, "horovod")

In [ ]:
%%time
mpi_run_python("horovod100.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp horovod100.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod100.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan 

In [ ]:
%%time
mpi_run_python("horovod200.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp horovod200.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod200.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [nan nan nan ... nan nan nan]
rank = 1, data = [nan nan nan ... nan nan nan]
rank = 2, data = [nan nan nan ... nan nan nan]

CPU times: user 3.33 ms, sys: 1.04 ms, total: 4.37 ms
Wall time: 9.26 s


In [ ]:
%%time
mpi_run_python("horovod400.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp horovod400.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod400.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 0, data = [nan nan nan ... nan nan nan]
rank = 1, data = [nan nan nan ... nan nan nan]
rank = 2, data = [nan nan nan ... nan nan nan]

CPU times: user 2.33 ms, sys: 2.89 ms, total: 5.22 ms
Wall time: 20.6 s


In [ ]:
%%time
mpi_run_python("horovod800.py", "--hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3")


 ### copy source code to node 34.68.45.15 ###
scp -i .ssh/id_gcp horovod800.py pilkiewiczkamil@34.68.45.15:./


 ### Running MPI ###
ssh -i .ssh/id_gcp pilkiewiczkamil@34.68.45.15 "mpiexec --hostfile hostfile_mpi -x MASTER_ADDR=bml-0 -x MASTER_PORT=12340 -n 3 python3 horovod800.py"
[Gloo] Rank 0 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 1 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
[Gloo] Rank 2 is connected to 2 peer ranks. Expected number of connected peer ranks is : 2
rank = 1, data = [nan nan nan ... nan nan nan]
rank = 2, data = [nan nan nan ... nan nan nan]
rank = 0, data = [nan nan nan ... nan nan nan]

CPU times: user 8.92 ms, sys: 1.59 ms, total: 10.5 ms
Wall time: 2min 16s


# IMPORTANT: Remove created resources at the end of the session
```
terraform destroy
```

**And don't forget to enter 'yes'!**

We can easily recreate them now, so nothing is lost upon deletion!